# Description Generation

This notebook performs description generation using a local LLM (Qwen 3
via Ollama).

Objectives:
1. Load the enriched homestay dataset.
2. Generate a factual description for each homestay using Qwen 3.
3. Save descriptions back into the same dataset file.


In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
from ollama import chat
from tqdm import tqdm
import os
import time

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================

INPUT_OUTPUT_FILE = "../data/processed/homestays_enriched.csv"

MODEL_NAME = "qwen3:4b"

CHECKPOINT_EVERY_N_ROWS = 10

FINAL_OUTPUT_COLUMN = "description"

CHECKPOINT_DIR = "../data/processed/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_FILE = f"{CHECKPOINT_DIR}/description_generation_checkpoint.csv"

In [ ]:
# ==========================================
# LOAD ENRICHED DATASET
# ==========================================

df = pd.read_csv(INPUT_OUTPUT_FILE)

print(f"Total Records: {len(df)}")
df.head()

In [ ]:
# ==========================================
# WRITING STYLE SELECTION
# ==========================================

def get_style(row):

    # Amenity-focused if the homestay has a mountain view
    if row.get("mountain_view", 0) == 1:
        return "amenity-focused"

    # Accessibility-focused if very close to town
    if row.get("town_proximity", "") == "Very Close":
        return "accessibility-focused"

    # Rating-focused if highly rated
    if float(row.get("rating", 0) or 0) >= 4.5:
        return "rating-focused"

    # Default style
    return "location-focused"

In [ ]:
# ==========================================
# PROMPT CONSTRUCTION
# ==========================================

AMENITY_COLUMNS = [
    "wifi", "parking", "breakfast", "mountain_view",
    "room_service", "bonfire_barbeque", "pickup_dropoff_service",
]

def create_prompt(row):

    style = get_style(row)

    amenities = [
        col.replace("_", " ").title()
        for col in AMENITY_COLUMNS
        if row.get(col, 0) == 1
    ]
    amenities_text = ", ".join(amenities) if amenities else "None listed"

    prompt = f"""
    Generate a factual homestay description.

    Writing Style: {style}

    Rules:
    - Use ONLY the supplied information.
    - Do NOT invent facts.
    - Do NOT mention facilities that are not listed.
    - Do NOT mention mountain views unless available.
    - You MUST explicitly state the category (Gold or Silver) somewhere in the description -- this is required, not optional.
    - Write AT LEAST 55 words and no more than 80 words. Reach this length by including MORE SPECIFIC real details already supplied (exact proximity to each of Deolo/Durpin/town, the specific village and block names, the exact rating and review count, every listed amenity) -- NOT by adding generic filler phrases like "a wonderful experience" or "nestled in the hills" or "a memorable stay." Every sentence must convey a specific fact from the data above.
    - Do not include word counts, notes, brackets, or explanations.
    - Return ONLY the description.

    Style Instructions:

    Location-focused:
    Start by describing where the homestay is situated.

    Amenity-focused:
    Highlight the facilities first.

    Accessibility-focused:
    Focus on proximity to town, Deolo, and Durpin.

    Rating-focused:
    Naturally mention ratings and reviews.

    Data:

    Name: {row.get('homestay_name', '')}
    Village: {row.get('village', '')}
    Block: {row.get('block', '')}
    Category: {row.get('category', '')}

    Rating: {row.get('rating', '')}
    Review Count: {row.get('review_count', '')}

    Distance to Town:
    {row.get('town_proximity', '')}

    Distance to Deolo:
    {row.get('deolo_proximity', '')}

    Distance to Durpin:
    {row.get('durpin_proximity', '')}

    Amenities:
    {amenities_text}
    """

    return prompt

In [ ]:
# ==========================================
# THINKING TRACE CLEANUP
# ==========================================

def strip_thinking(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()

In [ ]:
# ==========================================
# GENERATION LOOP
# ==========================================

def generate_descriptions(df, checkpoint_path=CHECKPOINT_FILE, checkpoint_every=CHECKPOINT_EVERY_N_ROWS):

    # Resume from checkpoint if one exists
    if os.path.exists(checkpoint_path):
        progress_df = pd.read_csv(checkpoint_path)
        descriptions = progress_df[FINAL_OUTPUT_COLUMN].tolist()
        print(f"Resuming: {len(descriptions)} rows already done.")
    else:
        descriptions = []
        print("No checkpoint -- starting from row 0.")

    start_row = len(descriptions)
    end_row = len(df)

    if start_row >= end_row:
        print("Already complete.")
        return descriptions

    start_time = time.time()

    for idx in tqdm(range(start_row, end_row), initial=start_row, total=end_row):

        row = df.iloc[idx]
        prompt = create_prompt(row)

        try:
            # num_ctx capped at 4096 -- prompt+response fit well under this;
            # Ollama was defaulting to 262144 and running mostly on CPU as a result
            response = chat(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                options={"num_ctx": 4096},
            )
            description = strip_thinking(response["message"]["content"])

        except Exception as e:
            # Save progress before re-raising so nothing generated so far is lost
            print(f"\nError at row {idx}: {e}")
            _save_checkpoint(df, descriptions, checkpoint_path)
            print("Checkpoint saved -- re-run to resume.")
            raise

        descriptions.append(description)

        # Periodic checkpoint + remaining-time estimate
        if (idx + 1) % checkpoint_every == 0:
            _save_checkpoint(df, descriptions, checkpoint_path)
            elapsed = time.time() - start_time
            avg_time = elapsed / (len(descriptions) - start_row)
            remaining = avg_time * (end_row - len(descriptions))
            print(f"\nRow {idx + 1}/{end_row} | avg {avg_time:.1f}s/row | ~{remaining/60:.1f} min left")

    _save_checkpoint(df, descriptions, checkpoint_path)
    print(f"\nDone: {len(descriptions)} descriptions.")
    return descriptions


def _save_checkpoint(df, descriptions, checkpoint_path):
    # Saves partial progress to disk so a run can resume after interruption
    temp_df = df.iloc[:len(descriptions)].copy()
    temp_df[FINAL_OUTPUT_COLUMN] = descriptions
    temp_df.to_csv(checkpoint_path, index=False)

In [ ]:
# ==========================================
# RUN GENERATION
# ==========================================

descriptions = generate_descriptions(df)
df["description"] = descriptions

In [ ]:
# ==========================================
# VALIDATION CHECKS
# ==========================================

# Check for any empty/failed descriptions
empty_count = (df["description"].str.strip() == "").sum()
print(f"Empty descriptions: {empty_count}")

# Word count distribution (target: 55-80 words)
word_counts = df["description"].str.split().str.len()
print("\nWord count distribution:")
print(word_counts.describe())

# Print a few samples for a manual quality check
print("\nSamples:")
for i in df.sample(min(3, len(df)), random_state=42).index:
    print(f"\n[{df.loc[i, 'homestay_name']}] ({get_style(df.loc[i])})")
    print(df.loc[i, "description"])

In [ ]:
# ==========================================
# SAVE ENRICHED DATASET WITH DESCRIPTIONS
# ==========================================

df.to_csv(INPUT_OUTPUT_FILE, index=False)

print("Description Generation Completed.")
print(f"Saved to: {INPUT_OUTPUT_FILE}")